<a href="https://colab.research.google.com/github/Apoorva-Menon/ExtremeYears/blob/milestone4/preprocessing/NASA_EDA_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import pandas as pd

In [2]:
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/nasa-viz/PSCompPars_2026.10.27_11.16.54.csv', skiprows=328)

Mounted at /content/drive


In [ ]:
df.head()

In [ ]:
pd.set_option("display.max_rows", None)

print(df.isna().sum())

In [ ]:
df_filtered = df[df["pl_controv_flag"] == 0].copy()

print(df_filtered.isna().sum())

In [ ]:
print("Total planets:", len(df))
print("Controversial planets:", len(df) - len(df_filtered))
print("Non-controversial planets:", len(df_filtered))

In [ ]:
selected_cols = [
    "pl_name",
    "hostname",
    "pl_rade",
    "pl_bmasse",
    "discoverymethod",
    "pl_eqt",
    "st_lum",
    "pl_orbeccen",
    "st_mass",
    "pl_orbper",
    "pl_orbsmax",
    "st_teff",
    "pl_insol"
]

missing = pd.DataFrame({
    "Available (Original)": df[selected_cols].notna().sum(),
    "Missing (Original)": df[selected_cols].isna().sum(),
    "Missing % (Original)": df[selected_cols].isna().mean() * 100,
    "Available (CF applied)": df_filtered[selected_cols].notna().sum(),
    "Missing (CF applied)": df_filtered[selected_cols].isna().sum(),
    "Missing % (CF applied)": df_filtered[selected_cols].isna().mean() * 100
})

display(missing.round(2))

## Shared preprocessing for Observable

Retain the existing non-controversial NASA planet rows and source columns. Standardize the numeric fields needed for the scientific metric and calculate irradiation_ratio (periastron irradiation / apastron irradiation). Missing or invalid eccentricity produces missing R, not an imputed value. No chart-specific filtering, binning, or ranking is performed in this pipeline.

In [ ]:
import numpy as np
from pathlib import Path

# Start from the original, non-controversial planet population.
planets = df_filtered.copy()

# Standardize the numeric inputs needed for the scientific calculation.
for column in ["pl_orbper", "pl_orbeccen"]:
    planets[column] = pd.to_numeric(planets[column], errors="coerce")
    planets.loc[~np.isfinite(planets[column]), column] = np.nan

# R is defined for bound elliptical orbits with 0 <= e < 1.
e = planets["pl_orbeccen"]
valid_e = e.notna() & e.ge(0) & e.lt(1)
planets["irradiation_ratio"] = np.nan
planets.loc[valid_e, "irradiation_ratio"] = (
    (1 + e.loc[valid_e]) / (1 - e.loc[valid_e])
) ** 2

# A source-recorded e=0 is retained as 0, yielding R=1.
# No orbital-period filter is applied here: charts may have different requirements.
print("Processed rows:", len(planets))
print("Duplicate planet names:", int(planets["pl_name"].duplicated().sum()))
display(planets[["pl_name", "pl_orbper", "pl_orbeccen", "irradiation_ratio"]].head())

### Preprocessing audit (not chart eligibility)

These checks describe source completeness and mathematical validity; they do not add flags to the exported CSV. Observable will determine eligible rows independently for each visualization.

In [ ]:
period = planets["pl_orbper"]
ratio = planets["irradiation_ratio"]
audit = pd.DataFrame([
    {"check": "Non-controversial planet rows", "count": len(planets)},
    {"check": "Finite, positive orbital periods", "count": int((period.notna() & period.gt(0)).sum())},
    {"check": "Valid eccentricities (0 <= e < 1)", "count": int(valid_e.sum())},
    {"check": "Finite irradiation ratios (R >= 1)", "count": int((ratio.notna() & np.isfinite(ratio) & ratio.ge(1)).sum())},
    {"check": "Recorded eccentricity e = 0", "count": int(e.eq(0).sum())},
    {"check": "Duplicate planet names", "count": int(planets["pl_name"].duplicated().sum())},
])
display(audit)

### Scientific spot-check

Inspect the largest calculated ratios to flag unusually high eccentricities for later scientific validation. This preview does not set the chart ranking or exclude any planet.

In [ ]:
display(
    planets.loc[planets["irradiation_ratio"].notna(),
                ["pl_name", "pl_orbper", "pl_orbeccen", "irradiation_ratio"]]
    .sort_values(["irradiation_ratio", "pl_name"], ascending=[False, True])
    .head(10)
)

### Export for Observable

Export the non-controversial NASA rows with their original source columns and the calculated irradiation_ratio. The CSV contains no chart-specific eligibility flags. In Observable, filter per chart, construct logarithmic histogram bins, rank planets, and implement tooltips and interactions.

In [ ]:
output_dir = Path("/content/extreme_years_exports") if Path("/content").exists() else Path("extreme_years_exports")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "planets_processed.csv"
planets.to_csv(output_path, index=False)
print("Saved:", output_path)
print("Rows:", len(planets), "Columns:", len(planets.columns))

# Optional Colab download:
# from google.colab import files
# files.download(str(output_path))

### Preprocessing required for Q3/Q4

In [ ]:
# Fill in missing orbit size and starlight using physics.
# pl_orbsmax_source column records where each value came from: archive or kepler_third_law

# Orbit size (AU) from Kepler's third law: a = cube root of (star mass × period in years²)
planets["pl_orbsmax_source"] = np.where(planets["pl_orbsmax"].notna(), "archive", None)
need_a = planets["pl_orbsmax"].isna() & planets["pl_orbper"].gt(0) & planets["st_mass"].gt(0)
planets.loc[need_a, "pl_orbsmax"] = np.cbrt(
    planets.loc[need_a, "st_mass"] * (planets.loc[need_a, "pl_orbper"] / 365.25) ** 2
)
planets.loc[need_a, "pl_orbsmax_source"] = "kepler_third_law"

In [ ]:

# pl_insol_source column records where each value came from: archive or from_st_lum
# Starlight at the average distance (Earth = 1) from the inverse-square law: S = L / a^2
# st_lum is stored as log10(L / L_sun), so 10 ** st_lum gives L in solar units.
planets["pl_insol_source"] = np.where(planets["pl_insol"].notna(), "archive", None)
need_s = planets["pl_insol"].isna() & planets["st_lum"].notna() & planets["pl_orbsmax"].gt(0)
planets.loc[need_s, "pl_insol"] = 10 ** planets.loc[need_s, "st_lum"] / planets.loc[need_s, "pl_orbsmax"] ** 2
planets.loc[need_s, "pl_insol_source"] = "from_st_lum"

In [ ]:
BURST_SHARE = 0.25          # below 25% of the year near peak is a burst year
EXCLUDE_UPPER_LIMITS = True # True = eccentricities reported only as limits are not usable

def above_half_peak_time(ecc):
    """Share of the year a planet spends at or above half of its peak starlight.
    Uses only eccentricity. e = 0 (circular) gives 1, i.e. the whole year."""
    ecc = np.asarray(ecc, dtype=float)
    share = np.ones_like(ecc)                                 # e = 0 -> whole year
    stretched = ecc > 0
    e_s = ecc[stretched]
    r_half = np.sqrt(2) * (1 - e_s)                             # distance where starlight = 1/2 peak (units of a)
    E_half = np.arccos(np.clip((1 - r_half) / e_s, -1, 1))      # orbit position at that distance
    share[stretched] = (E_half - e_s * np.sin(E_half)) / np.pi   # Kepler's equation
    return share

In [ ]:
e = planets["pl_orbeccen"]
valid_e = e.notna() & e.ge(0) & e.lt(1)
if EXCLUDE_UPPER_LIMITS:
    valid_e &= planets["pl_orbeccenlim"].fillna(0).eq(0)
insol = planets["pl_insol"]

In [ ]:
# Eccentricity with its filters
planets["e"] = e.where(valid_e)

# Quality label - defines whether the eccentricity is 0 (which means it was not measured) or if it is measured
planets["e_quality"] = np.select(
    [valid_e & e.gt(0), valid_e & e.eq(0)],
    ["measured", "recorded_zero"],
    default="not_usable")

In [ ]:
# Peak starlight at closest approach: S_peak = pl_insol / (1 − e)^2
has_peak = valid_e & insol.gt(0)
planets["S_peak"] = np.nan
planets.loc[has_peak, "S_peak"] = insol[has_peak] / (1 - e[has_peak]) ** 2

In [ ]:
# Share of the year near peak, days near peak, and burst / prolonged label
planets["above_half_peak_time"] = np.nan
planets.loc[valid_e, "above_half_peak_time"] = above_half_peak_time(e[valid_e])
planets["days_bright"] = planets["above_half_peak_time"] * planets["pl_orbper"]
planets["exposure"] = None
planets.loc[valid_e, "exposure"] = np.where(
    planets.loc[valid_e, "above_half_peak_time"] < BURST_SHARE, "burst", "prolonged"
)

In [ ]:
display(planets.loc[planets["e_quality"] != "not_usable",
        ["pl_name", "pl_orbeccen", "e_quality", "pl_insol", "S_peak",
         "above_half_peak_time", "days_bright", "exposure"]].head())

In [ ]:
STORY_COLS = ["pl_name", "hostname", "discoverymethod", "pl_orbper", "pl_insol",
              "e", "e_quality", "S_peak", "irradiation_ratio",
              "above_half_peak_time", "days_bright", "exposure"]

story = planets.loc[valid_e, STORY_COLS].round(6)
story_path = output_dir / "planets_story.csv"
story.to_csv(story_path, index=False)

print("Saved:", story_path)
print(f"Rows: {len(story):,}   measured: {(story.e_quality == 'measured').sum():,}   recorded zero: {(story.e_quality == 'recorded_zero').sum():,}")

from google.colab import files
files.download(str(story_path))

In [ ]:
# Run after the Q3/Q4 cells have filled orbit size/flux and calculated bright days.
# Keep their shared planets table unchanged.
q56_e = planets["pl_orbeccen"]
q56_eligible = (
    planets["pl_orbper"].gt(0)
    & planets["pl_orbsmax"].gt(0)
    & planets["pl_insol"].gt(0)
    & q56_e.ge(0) & q56_e.lt(1)
    & planets["pl_orbeccenlim"].eq(0)
)
q56_base = planets.loc[q56_eligible].copy()
print(f"Q5/Q6 eligible planets: {len(q56_base):,}")

## Q5/Q6 — Characteristics and extreme planetary years

**Q5:** How do planetary, orbital, and stellar characteristics relate to irradiation extremes?

**Q6:** Which five exoplanets have the wildest years, and what makes each extraordinary?

We use irradiation contrast R = ((1 + e) / (1 − e))². This measures variation in incoming starlight, not temperature or habitability. These cells prepare the data for the four Q5/Q6 D3 charts in Observable.

In [ ]:
q56_shared_export = planets.copy(deep=True)

### 1. Select the Q5/Q6 population
Reuse the shared preprocessing’s filled orbit sizes and reference fluxes. Retain positive period, orbit size, and flux, with 0 ≤ eccentricity < 1 and an eccentricity limit flag of zero

In [ ]:
# Run after the Q3/Q4 cells have filled orbit size/flux and calculated bright days.
# Keep their shared planets table unchanged.
q56_e = planets["pl_orbeccen"]
q56_eligible = (
    planets["pl_orbper"].gt(0)
    & planets["pl_orbsmax"].gt(0)
    & planets["pl_insol"].gt(0)
    & q56_e.ge(0) & q56_e.lt(1)
    & planets["pl_orbeccenlim"].eq(0)
)
q56_base = planets.loc[q56_eligible].copy()
print(f"Q5/Q6 eligible planets: {len(q56_base):,}")

### 2. Prepare the chart fields
Reuse the existing R, peak flux, and bright days. Add luminosity in solar units, minimum flux, and R at valid published eccentricity-error endpoints. Missing or invalid error bounds remain blank; these are not rank confidence intervals.

In [ ]:
q56_e = q56_base["pl_orbeccen"]

# The existing cells already supply R, S_peak, and days_bright.
q56_base["lum_solar"] = 10 ** q56_base["st_lum"]
q56_base["S_min"] = q56_base["pl_insol"] / (1 + q56_e) ** 2

# Transform valid published error endpoints; leave unavailable bounds blank.
q56_lower_e = q56_e + q56_base["pl_orbeccenerr2"]
q56_upper_e = q56_e + q56_base["pl_orbeccenerr1"]
q56_valid_lower = q56_lower_e.ge(0) & q56_lower_e.le(q56_e)
q56_valid_upper = q56_upper_e.ge(q56_e) & q56_upper_e.lt(1)

q56_base["R_lower"] = np.nan
q56_base["R_upper"] = np.nan
q56_base.loc[q56_valid_lower, "R_lower"] = (
    (1 + q56_lower_e.loc[q56_valid_lower]) / (1 - q56_lower_e.loc[q56_valid_lower])
) ** 2
q56_base.loc[q56_valid_upper, "R_upper"] = (
    (1 + q56_upper_e.loc[q56_valid_upper]) / (1 - q56_upper_e.loc[q56_valid_upper])
) ** 2

### 3. Verify the results
Check unique planets, the flux-ratio identity, and the circular-orbit case. Show the actual five highest central R values and their inputs. With this source export, expect 5,018 eligible planets: 1,876 positive eccentricities and 3,142 recorded zeros.

In [ ]:
q56_positive_count = int((q56_base["pl_orbeccen"] > 0).sum())
q56_zero_count = int((q56_base["pl_orbeccen"] == 0).sum())
print(f"Q3 comparison: {q56_positive_count:,} positive eccentricities; {q56_zero_count:,} recorded zeros")

assert q56_base["pl_name"].is_unique
assert q56_positive_count + q56_zero_count == len(q56_base)
assert np.allclose(
    q56_base["S_peak"] / q56_base["S_min"],
    q56_base["irradiation_ratio"]
)
assert q56_base["above_half_peak_time"].between(0, 1).all()
q56_circular = q56_base["pl_orbeccen"].eq(0)
assert q56_base.loc[q56_circular, "irradiation_ratio"].eq(1).all()
assert q56_base.loc[q56_circular, "above_half_peak_time"].eq(1).all()

q56_ranked = q56_base.sort_values(
    ["irradiation_ratio", "pl_name"], ascending=[False, True]
).head(5)
q56_sample_columns = [
    "pl_name", "pl_orbeccen", "pl_orbeccenerr2", "pl_orbeccenerr1",
    "pl_orbper", "pl_orbsmax", "irradiation_ratio", "R_lower", "R_upper", "days_bright"
]
display(q56_ranked[q56_sample_columns])

In [ ]:
q56_shared_export = planets.copy(deep=True)
q56_lookup = q56_base.set_index("pl_name")

q56_shared_export["q56_eligible"] = (
    q56_shared_export["pl_name"].isin(q56_lookup.index).astype(int)
)

for q56_column in ["lum_solar", "R_lower", "R_upper"]:
    q56_shared_export[q56_column] = (
        q56_shared_export["pl_name"].map(q56_lookup[q56_column])
    )

q56_shared_export.to_csv(output_path, index=False)

from google.colab import files as q56_files
q56_files.download(str(output_path))